# Shapesphysical Model Results

This notebook compares the finished cleaned `shapesphysical` structure-comparison runs for:
- `Gemma 2 2B`
- `Gemma 2 9B`
- `Llama 3.1 8B`

It is designed for quick thesis-oriented inspection of:
- family-level brain and LM prediction metrics
- feature-importance correlation
- sample RSA
- top parcels, LM targets, and shared predictors

The notebook reads the completed outputs under `structure_comparison/outputs/`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "structure_comparison").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing structure_comparison/")


ROOT = find_repo_root()
OUTPUT_ROOT = ROOT / "structure_comparison" / "outputs"
BRAIN_TARGETS = ROOT / "structure_comparison" / "brain_targets" / "shapesphysical_schaefer200_cleaned_batches00_05.npz"

RUNS = {
    "gemma_2_2b": OUTPUT_ROOT / "final_output_retry_gemma_2_2b_shapesphysical_cleaned_batches00_05",
    "gemma_2_9b": OUTPUT_ROOT / "final_output_retry_gemma_2_9b_shapesphysical_cleaned_batches00_05",
    "llama_3_1_8b": OUTPUT_ROOT / "final_output_retry_llama_3_1_8b_shapesphysical_cleaned_batches00_05",
}

DISPLAY_NAMES = {
    "gemma_2_2b": "Gemma 2 2B",
    "gemma_2_9b": "Gemma 2 9B",
    "llama_3_1_8b": "Llama 3.1 8B",
}

for path in list(RUNS.values()) + [BRAIN_TARGETS]:
    if not path.exists():
        raise FileNotFoundError(path)


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


analysis_by_model = {
    model_name: load_json(run_dir / "analysis_summary.json")
    for model_name, run_dir in RUNS.items()
}

family_rows = []
for model_name, analysis in analysis_by_model.items():
    for family in analysis["families"]:
        family_rows.append(
            {
                "model": model_name,
                "model_display": DISPLAY_NAMES[model_name],
                "family": family["family_name"],
                "predictor_count": family["predictor_count"],
                "lm_target_count": family["lm_target_count"],
                "brain_sample_count": family["brain_sample_count"],
                "brain_run_count": family["brain_run_count"],
                "brain_mean_test_correlation": family["brain_mean_test_correlation"],
                "brain_mean_test_r2": family["brain_mean_test_r2"],
                "lm_mean_test_correlation": family["lm_mean_test_correlation"],
                "lm_mean_test_r2": family["lm_mean_test_r2"],
                "feature_importance_correlation": family["feature_importance_correlation"],
                "sample_rsa_correlation": family["sample_rsa_correlation"],
                "brain_consensus_alpha": family["brain_consensus_alpha"],
                "lm_consensus_alpha": family["lm_consensus_alpha"],
            }
        )

family_df = pd.DataFrame(family_rows).sort_values(["model_display", "family"]).reset_index(drop=True)
brain_bundle = np.load(BRAIN_TARGETS, allow_pickle=False)
brain_shape = tuple(int(x) for x in brain_bundle["values"].shape)


In [ ]:
PALETTE = {
    "Gemma 2 2B": "#1f77b4",
    "Gemma 2 9B": "#ff7f0e",
    "Llama 3.1 8B": "#2ca02c",
}


def _format_metric(value: float) -> str:
    if pd.isna(value):
        return "NaN"
    return f"{value:.3f}"


def html_bar_chart(
    df: pd.DataFrame,
    label_col: str,
    value_col: str,
    title: str,
    color_col: str | None = None,
    width_px: int = 720,
) -> HTML:
    plot_df = df.copy()
    max_abs = float(plot_df[value_col].abs().max()) if not plot_df.empty else 1.0
    if max_abs == 0:
        max_abs = 1.0
    rows = []
    for _, row in plot_df.iterrows():
        label = str(row[label_col])
        value = float(row[value_col])
        width = max(2, int((abs(value) / max_abs) * 420))
        color_key = str(row[color_col]) if color_col else None
        color = PALETTE.get(color_key, "#4c78a8")
        sign = "negative" if value < 0 else "positive"
        rows.append(
            f'''
            <div style="display:flex; align-items:center; gap:12px; margin:6px 0;">
              <div style="width:210px; font-family:monospace; font-size:12px;">{label}</div>
              <div style="flex:1; background:#f3f4f6; border-radius:6px; height:18px; position:relative;">
                <div style="width:{width}px; height:18px; background:{color}; opacity:{0.85 if sign == "positive" else 0.55}; border-radius:6px;"></div>
              </div>
              <div style="width:72px; text-align:right; font-family:monospace; font-size:12px;">{_format_metric(value)}</div>
            </div>
            '''
        )
    html = f'''
    <div style="width:{width_px}px; max-width:100%;">
      <div style="font-weight:700; margin:8px 0 12px 0;">{title}</div>
      {''.join(rows)}
    </div>
    '''
    return HTML(html)


def html_heatmap_table(
    df: pd.DataFrame,
    title: str,
    cmap_low: str = "#fef3c7",
    cmap_high: str = "#1d4ed8",
) -> HTML:
    numeric_df = df.copy()
    values = numeric_df.to_numpy(dtype=float)
    finite = values[np.isfinite(values)]
    vmin = float(finite.min()) if finite.size else 0.0
    vmax = float(finite.max()) if finite.size else 1.0
    if vmax == vmin:
        vmax = vmin + 1.0

    def mix_color(value: float) -> str:
        if not np.isfinite(value):
            return "#f3f4f6"
        ratio = (value - vmin) / (vmax - vmin)
        ratio = max(0.0, min(1.0, ratio))
        low = (254, 243, 199)
        high = (29, 78, 216)
        rgb = tuple(int(low[i] + ratio * (high[i] - low[i])) for i in range(3))
        return f"rgb({rgb[0]}, {rgb[1]}, {rgb[2]})"

    header = "".join(f"<th style='padding:6px 10px; text-align:right;'>{col}</th>" for col in numeric_df.columns)
    body_rows = []
    for index, row in numeric_df.iterrows():
        cells = []
        for value in row:
            bg = mix_color(float(value))
            text_color = "#ffffff" if np.isfinite(value) and float(value) > (vmin + vmax) / 2 else "#111827"
            cells.append(
                f"<td style='padding:6px 10px; text-align:right; background:{bg}; color:{text_color}; font-family:monospace;'>{_format_metric(float(value))}</td>"
            )
        body_rows.append(f"<tr><th style='padding:6px 10px; text-align:left;'>{index}</th>{''.join(cells)}</tr>")

    html = f'''
    <div style="margin-top:12px;">
      <div style="font-weight:700; margin:8px 0 12px 0;">{title}</div>
      <table style="border-collapse:collapse; font-size:12px;">
        <thead><tr><th></th>{header}</tr></thead>
        <tbody>{''.join(body_rows)}</tbody>
      </table>
    </div>
    '''
    return HTML(html)


In [ ]:
display(
    Markdown(
        "\n".join(
            [
                "## Snapshot",
                f"- cleaned brain bundle: `{BRAIN_TARGETS}`",
                f"- cleaned brain matrix shape: `{brain_shape}`",
                f"- models compared: `{', '.join(DISPLAY_NAMES.values())}`",
                f"- total family runs summarized: `{family_df.shape[0]}`",
            ]
        )
    )
)

display(Markdown("## Model Output Directories"))
display(
    pd.DataFrame(
        [{"model": DISPLAY_NAMES[key], "path": str(value)} for key, value in RUNS.items()]
    )
)


In [ ]:
metric_columns = [
    "brain_mean_test_correlation",
    "brain_mean_test_r2",
    "lm_mean_test_correlation",
    "lm_mean_test_r2",
    "feature_importance_correlation",
    "sample_rsa_correlation",
]

display(Markdown("## Family-Level Metrics"))
display(
    family_df[
        [
            "model_display",
            "family",
            "predictor_count",
            "lm_target_count",
            *metric_columns,
        ]
    ].sort_values(["model_display", "family"])
)


In [ ]:
all_layers_df = family_df[family_df["family"] == "all_layers"].copy()
display(Markdown("## All-Layers Comparison"))
display(
    all_layers_df[
        [
            "model_display",
            "predictor_count",
            "lm_target_count",
            "brain_mean_test_correlation",
            "brain_mean_test_r2",
            "lm_mean_test_correlation",
            "lm_mean_test_r2",
            "feature_importance_correlation",
            "sample_rsa_correlation",
        ]
    ].sort_values("brain_mean_test_correlation", ascending=False)
)


In [ ]:
display(Markdown("## Plots: All-Layers Brain Correlation"))
display(
    html_bar_chart(
        all_layers_df.sort_values("brain_mean_test_correlation", ascending=False),
        label_col="model_display",
        value_col="brain_mean_test_correlation",
        title="All-Layers Brain Mean Test Correlation",
        color_col="model_display",
    )
)

display(Markdown("## Plots: All-Layers Feature-Importance Correlation"))
display(
    html_bar_chart(
        all_layers_df.sort_values("feature_importance_correlation", ascending=False),
        label_col="model_display",
        value_col="feature_importance_correlation",
        title="All-Layers Feature-Importance Correlation",
        color_col="model_display",
    )
)


In [ ]:
best_brain_df = (
    family_df.sort_values(["model_display", "brain_mean_test_correlation"], ascending=[True, False])
    .groupby("model_display", as_index=False)
    .first()
)
best_feature_df = (
    family_df.sort_values(["model_display", "feature_importance_correlation"], ascending=[True, False])
    .groupby("model_display", as_index=False)
    .first()
)

display(Markdown("## Best Family Per Model"))
display(
    best_brain_df[
        [
            "model_display",
            "family",
            "brain_mean_test_correlation",
            "brain_mean_test_r2",
            "feature_importance_correlation",
            "sample_rsa_correlation",
        ]
    ].rename(columns={"family": "best_family_by_brain_correlation"})
)
display(
    best_feature_df[
        [
            "model_display",
            "family",
            "feature_importance_correlation",
            "brain_mean_test_correlation",
            "sample_rsa_correlation",
        ]
    ].rename(columns={"family": "best_family_by_feature_importance"})
)


In [ ]:
display(Markdown("## Plots: Best Brain Family Per Model"))
display(
    html_bar_chart(
        best_brain_df.sort_values("brain_mean_test_correlation", ascending=False),
        label_col="model_display",
        value_col="brain_mean_test_correlation",
        title="Best Family Brain Correlation by Model",
        color_col="model_display",
    )
)

display(Markdown("## Plots: Best Feature-Importance Family Per Model"))
display(
    html_bar_chart(
        best_feature_df.sort_values("feature_importance_correlation", ascending=False),
        label_col="model_display",
        value_col="feature_importance_correlation",
        title="Best Family Feature-Importance Correlation by Model",
        color_col="model_display",
    )
)


In [ ]:
def load_family_summary(model_name: str, family_name: str) -> dict:
    return load_json(RUNS[model_name] / family_name / "summary.json")


def flatten_ranked_items(model_name: str, family_name: str, key: str, top_k: int = 10) -> pd.DataFrame:
    summary = load_family_summary(model_name, family_name)
    rows = []
    for rank, item in enumerate(summary.get(key, [])[:top_k], start=1):
        row = {"rank": rank, "model": DISPLAY_NAMES[model_name], "family": family_name}
        row.update(item)
        rows.append(row)
    return pd.DataFrame(rows)


all_layer_parcels = pd.concat(
    [flatten_ranked_items(model_name, "all_layers", "top_parcels", top_k=10) for model_name in RUNS],
    ignore_index=True,
)
all_layer_targets = pd.concat(
    [flatten_ranked_items(model_name, "all_layers", "top_lm_targets", top_k=10) for model_name in RUNS],
    ignore_index=True,
)
all_layer_predictors = pd.concat(
    [flatten_ranked_items(model_name, "all_layers", "top_shared_predictors", top_k=15) for model_name in RUNS],
    ignore_index=True,
)

display(Markdown("## All-Layers Top Parcels"))
display(all_layer_parcels)
display(Markdown("## All-Layers Top LM Targets"))
display(all_layer_targets)
display(Markdown("## All-Layers Top Shared Predictors"))
display(all_layer_predictors)


In [ ]:
predictor_counts = (
    all_layer_predictors.groupby("feature_name")
    .size()
    .reset_index(name="models_appearing")
    .sort_values(["models_appearing", "feature_name"], ascending=[False, True])
)

parcel_counts = (
    all_layer_parcels.groupby("target_name")
    .size()
    .reset_index(name="models_appearing")
    .sort_values(["models_appearing", "target_name"], ascending=[False, True])
)

display(Markdown("## Recurring Shared Predictors Across Models"))
display(predictor_counts.head(20))
display(Markdown("## Recurring Top Parcels Across Models"))
display(parcel_counts.head(20))


In [ ]:
metric_pivot = family_df.pivot_table(
    index="family",
    columns="model_display",
    values=["brain_mean_test_correlation", "lm_mean_test_correlation", "feature_importance_correlation", "sample_rsa_correlation"],
)
display(Markdown("## Metric Pivot"))
display(metric_pivot.sort_index())


In [ ]:
brain_corr_pivot = family_df.pivot_table(
    index="family",
    columns="model_display",
    values="brain_mean_test_correlation",
)
feature_corr_pivot = family_df.pivot_table(
    index="family",
    columns="model_display",
    values="feature_importance_correlation",
)
rsa_pivot = family_df.pivot_table(
    index="family",
    columns="model_display",
    values="sample_rsa_correlation",
)

display(Markdown("## Plot: Brain Correlation Heatmap"))
display(html_heatmap_table(brain_corr_pivot.sort_index(), "Brain Mean Test Correlation by Family"))

display(Markdown("## Plot: Feature-Importance Correlation Heatmap"))
display(html_heatmap_table(feature_corr_pivot.sort_index(), "Feature-Importance Correlation by Family"))

display(Markdown("## Plot: Sample RSA Heatmap"))
display(html_heatmap_table(rsa_pivot.sort_index(), "Sample RSA Correlation by Family"))


In [ ]:
top_predictor_strength = (
    all_layer_predictors.groupby(["model", "feature_name"], as_index=False)["brain_importance", "lm_importance"]
    .mean()
)
top_predictor_strength["combined_importance"] = (
    top_predictor_strength["brain_importance"] + top_predictor_strength["lm_importance"]
)

plot_predictors = (
    top_predictor_strength.sort_values(["model", "combined_importance"], ascending=[True, False])
    .groupby("model", as_index=False)
    .head(8)
    .copy()
)
plot_predictors["label"] = plot_predictors["model"] + " | " + plot_predictors["feature_name"]

display(Markdown("## Plot: Top Shared Predictors"))
display(
    html_bar_chart(
        plot_predictors.sort_values("combined_importance", ascending=False),
        label_col="label",
        value_col="combined_importance",
        title="Top Shared Predictors by Combined Brain + LM Importance",
        color_col="model",
        width_px=920,
    )
)


In [ ]:
display(
    Markdown(
        "\n".join(
            [
                "## Notes",
                "- Each model uses its own transcript tokenization and its own native layer set.",
                "- The families are therefore comparable within-model and descriptively across models, but layer numbers are not directly aligned across architectures.",
                "- `all_layers` is the pooled family over the selected layers available in that model run.",
            ]
        )
    )
)
